# 03 — Inter-Annotator Agreement (IAA)
**Task:** Sentiment Analysis  
**Metrics covered:** Cohen's κ, Fleiss' κ, Krippendorff's α, percentage agreement, confusion heatmaps

---

In [ ]:
!pip install -q pandas numpy scikit-learn krippendorff seaborn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import krippendorff

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABELS = ['Negative', 'Neutral', 'Positive']

## 1. Simulated Annotation Data
Three annotators label 200 movie reviews for sentiment (0=Neg, 1=Neu, 2=Pos).

In [ ]:
np.random.seed(42)
N = 200

# Ground truth labels
true_labels = np.random.choice([0, 1, 2], size=N, p=[0.35, 0.25, 0.40])

def noisy_annotator(true, noise=0.1):
    """Simulate annotator with given error rate."""
    labels = true.copy()
    flip_idx = np.random.rand(len(labels)) < noise
    labels[flip_idx] = np.random.choice([0, 1, 2], size=flip_idx.sum())
    return labels

ann_A = noisy_annotator(true_labels, noise=0.08)  # Expert
ann_B = noisy_annotator(true_labels, noise=0.15)  # Moderate
ann_C = noisy_annotator(true_labels, noise=0.25)  # Novice

df = pd.DataFrame({
    'item_id': range(N),
    'annotator_A': ann_A,
    'annotator_B': ann_B,
    'annotator_C': ann_C,
})
df.head(10)

## 2. Percentage Agreement (naïve baseline)

In [ ]:
def pct_agreement(a, b):
    return np.mean(np.array(a) == np.array(b))

pairs = [('A','B'), ('A','C'), ('B','C')]
for x, y in pairs:
    pct = pct_agreement(df[f'annotator_{x}'], df[f'annotator_{y}'])
    print(f"  {x} vs {y}: {pct:.3f} ({pct*100:.1f}%)")

## 3. Cohen's κ (pairwise)

In [ ]:
def kappa_interpretation(k):
    if k < 0:     return 'Poor (< 0)'
    elif k < 0.2: return 'Slight'
    elif k < 0.4: return 'Fair'
    elif k < 0.6: return 'Moderate'
    elif k < 0.8: return 'Substantial'
    else:         return 'Almost Perfect'

print("Pairwise Cohen's κ:")
kappa_results = {}
for x, y in pairs:
    k = cohen_kappa_score(df[f'annotator_{x}'], df[f'annotator_{y}'])
    kappa_results[f'{x}-{y}'] = k
    print(f"  {x} vs {y}: κ = {k:.4f}  [{kappa_interpretation(k)}]")

## 4. Weighted Cohen's κ (ordinal labels)

In [ ]:
print("Weighted Cohen's κ (linear weights — ordinal):")
for x, y in pairs:
    k = cohen_kappa_score(df[f'annotator_{x}'], df[f'annotator_{y}'], weights='linear')
    print(f"  {x} vs {y}: κ_w = {k:.4f}  [{kappa_interpretation(k)}]")

## 5. Fleiss' κ (multi-rater)

In [ ]:
def fleiss_kappa(ratings_matrix):
    """
    ratings_matrix: (N_items, N_categories) — counts of raters per category per item.
    """
    n_items, n_cats = ratings_matrix.shape
    n_raters = ratings_matrix.sum(axis=1)[0]  # assumes same number of raters per item
    
    # Proportion per category
    p_j = ratings_matrix.sum(axis=0) / (n_items * n_raters)
    
    # Per-item agreement
    P_i = ((ratings_matrix**2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    P_e   = (p_j**2).sum()
    
    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa

# Build ratings matrix: for each item, count how many raters chose each label
ann_matrix = np.column_stack([
    df['annotator_A'].values,
    df['annotator_B'].values,
    df['annotator_C'].values,
])  # shape (N, 3)

# Convert to category-count matrix (N, 3 categories)
cat_counts = np.apply_along_axis(
    lambda row: np.bincount(row, minlength=3), axis=1, arr=ann_matrix
)

fk = fleiss_kappa(cat_counts)
print(f"Fleiss' κ (all 3 annotators): {fk:.4f}  [{kappa_interpretation(fk)}]")

## 6. Krippendorff's α

In [ ]:
# krippendorff expects shape (n_annotators, n_items)
reliability_data = np.array([
    df['annotator_A'].values,
    df['annotator_B'].values,
    df['annotator_C'].values,
], dtype=float)

alpha_nominal = krippendorff.alpha(reliability_data, level_of_measurement='nominal')
alpha_ordinal = krippendorff.alpha(reliability_data, level_of_measurement='ordinal')

print(f"Krippendorff's α (nominal): {alpha_nominal:.4f}")
print(f"Krippendorff's α (ordinal): {alpha_ordinal:.4f}")
print("\nRule of thumb: α ≥ 0.80 for reliable data, α ≥ 0.667 for tentative.")

## 7. Disagreement Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (x, y) in zip(axes, pairs):
    cm = confusion_matrix(df[f'annotator_{x}'], df[f'annotator_{y}'], labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False)
    k = kappa_results[f'{x}-{y}']
    ax.set_title(f'Annotator {x} vs {y}\nκ = {k:.3f}', fontsize=12)
    ax.set_xlabel(f'Annotator {y}')
    ax.set_ylabel(f'Annotator {x}')

plt.suptitle('Confusion Matrices Between Annotators', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Summary Table & Gold Labels

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        "% Agreement A-B", "% Agreement A-C", "% Agreement B-C",
        "Cohen's κ A-B", "Cohen's κ A-C", "Cohen's κ B-C",
        "Fleiss' κ",
        "Krippendorff α (nominal)", "Krippendorff α (ordinal)"
    ],
    'Score': [
        pct_agreement(ann_A, ann_B), pct_agreement(ann_A, ann_C), pct_agreement(ann_B, ann_C),
        *[kappa_results[f'{x}-{y}'] for x, y in pairs],
        fk, alpha_nominal, alpha_ordinal
    ]
})
summary['Score'] = summary['Score'].round(4)
summary


In [ ]:
# Majority-vote gold labels
from scipy.stats import mode

gold = mode(ann_matrix, axis=1).mode.flatten()
df['gold_label'] = gold
print("Gold label distribution:")
print(df['gold_label'].map(LABEL_MAP).value_counts())

# Save for downstream notebooks
df.to_csv('/tmp/sentiment_annotated.csv', index=False)
print("\nSaved to /tmp/sentiment_annotated.csv")

---
**Next Notebook →** `04_eda_train_test_split.ipynb`